# Sprint 4B Class-Aware Extension

Thin Colab runner. Core logic stays in `src/dl_midterm/` and `scripts/`. Run full fusion only after validation-based stop/go passes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
%%bash
set -euo pipefail
BRANCH="${SPRINT4B_BRANCH:-codex/sprint4b-colab-run}"
cd /content
if [ ! -d dl-assignment/.git ]; then
  git clone https://github.com/KasimDeliaci/dl-midterm-project.git dl-assignment
fi
cd /content/dl-assignment
git fetch origin "$BRANCH"
git checkout -B "$BRANCH" "origin/$BRANCH"
python -m pip install -q uv
uv sync

In [ ]:
%%bash
set -euo pipefail
cd /content/dl-assignment
BUNDLE=""
for candidate in \
  /content/drive/MyDrive/ham10000_colab_bundle.tar \
  "/content/drive/MyDrive/Colab Notebooks/ham10000_colab_bundle.tar" \
  /content/drive/MyDrive/dl-assignment/ham10000_colab_bundle.tar; do
  if [ -f "$candidate" ]; then BUNDLE="$candidate"; break; fi
done
if [ -z "$BUNDLE" ]; then
  echo "HAM10000 bundle not found on Drive" >&2
  exit 1
fi
tar -xf "$BUNDLE" -C /content/dl-assignment
mkdir -p /content/drive/MyDrive/dl-midterm-artifacts

In [ ]:
%%bash
set -euo pipefail
cd /content/dl-assignment
uv run python scripts/finetune_backbone.py \
  --config configs/experiments/sprint4b_classaware_backbones.yaml \
  --default-config configs/default.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --feature-source finetuned_classaware \
  --batch-size 32

In [ ]:
%%bash
set -euo pipefail
cd /content/dl-assignment
uv run python scripts/train_mlp.py \
  --config configs/experiments/sprint4b_classaware_feature_matrix.yaml \
  --default-config configs/default.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --feature-source finetuned_classaware

In [ ]:
%%bash
set -euo pipefail
cd /content/dl-assignment
uv run python scripts/finetune_backbone.py \
  --config configs/experiments/sprint4b_deeper_screen.yaml \
  --default-config configs/default.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --feature-source finetuned_deeper \
  --backbone resnet50 \
  --batch-size 32
uv run python scripts/train_mlp.py \
  --config configs/experiments/sprint4b_deeper_screen.yaml \
  --default-config configs/default.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --feature-source finetuned_deeper

## Optional Full Class-Aware Matrix

Run this cell only if screening passes the validation macro-F1 stop/go rule.

In [ ]:
%%bash
set -euo pipefail
cd /content/dl-assignment
uv run python scripts/run_experiment_matrix.py \
  --config configs/experiments/sprint4b_classaware_feature_matrix.yaml \
  --default-config configs/default.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --feature-source finetuned_classaware

In [ ]:
%%bash
set -euo pipefail
cd /content/dl-assignment
uv run python scripts/make_report_assets.py \
  --feature-source finetuned_classaware \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --feature-root artifacts/features
rsync -a artifacts/report_assets/ /content/drive/MyDrive/dl-midterm-artifacts/report_assets/
rsync -a artifacts/features/ham10000/finetuned_classaware/ /content/drive/MyDrive/dl-midterm-artifacts/features/ham10000/finetuned_classaware/
rsync -a artifacts/features/ham10000/finetuned_deeper/ /content/drive/MyDrive/dl-midterm-artifacts/features/ham10000/finetuned_deeper/
rsync -a artifacts/checkpoints/finetuned_classaware_backbones/ /content/drive/MyDrive/dl-midterm-artifacts/checkpoints/finetuned_classaware_backbones/
rsync -a artifacts/checkpoints/finetuned_deeper_backbones/ /content/drive/MyDrive/dl-midterm-artifacts/checkpoints/finetuned_deeper_backbones/